In [1]:
%load_ext autoreload
%autoreload 2

In [2]:

from pyspark_data_provenance import (
    data_provenance_enabled, 
    build_data_provenance_session
)

In [3]:
from pyspark.sql import SparkSession


spark = (
    # SparkSession
    # .builder
    build_data_provenance_session()
    .appName("data-provenance-notebook")
    .getOrCreate()
)


26/04/23 16:12:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
print(spark.conf.get("spark.provenance.enabled", "false"))

with data_provenance_enabled(spark):
    print(spark.conf.get("spark.provenance.enabled", "false"))

print(spark.conf.get("spark.provenance.enabled", "false"))

false
true
false


## Create toy dataframes

In [5]:
from datetime import date

df = spark.createDataFrame([
    ("A", date(2026, 1, 15), 10.0, 90),
    ("A", date(2026, 1, 16), 10.0, 120),
    ("A", date(2026, 1, 17), 5.0, 300),
    ("B", date(2026, 1, 15), 100.0, 20),
    ("B", date(2026, 1, 16), 100.0, 30),
    ("C", date(2026, 1, 17), 80.0, 60),
    ("F", date(2026, 1, 16), 50.0, 70)
], ["product", "date", "price", "quantity"]
)
df.printSchema()

df.toPandas()

root
 |-- product: string (nullable = true)
 |-- date: date (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: long (nullable = true)



,product,date,price,quantity
0,A,2026-01-15,10.0,90
1,A,2026-01-16,10.0,120
2,A,2026-01-17,5.0,300
3,B,2026-01-15,100.0,20
4,B,2026-01-16,100.0,30
5,C,2026-01-17,80.0,60
6,F,2026-01-16,50.0,70


In [6]:
df2 = spark.createDataFrame([
    ("A", "Bike"),
    ("B", "Handball"),
    ("C", "Bike"),
    ("D", "Handball"),
    ("E", "Running")
],["letter","labell"]
)

df2.printSchema()
df2.toPandas()



root
 |-- letter: string (nullable = true)
 |-- labell: string (nullable = true)



,letter,labell
0,A,Bike
1,B,Handball
2,C,Bike
3,D,Handball
4,E,Running


## Tests with data_provenance_enabled

### Select

In [7]:
with data_provenance_enabled(spark):
    df3 = df.select("*")
df3.toPandas()

,product,date,price,quantity,_provenance_tag
0,A,2026-01-15,10.0,90,LogicalRDD_8589934592
1,A,2026-01-16,10.0,120,LogicalRDD_17179869184
2,A,2026-01-17,5.0,300,LogicalRDD_34359738368
3,B,2026-01-15,100.0,20,LogicalRDD_42949672960
4,B,2026-01-16,100.0,30,LogicalRDD_60129542144
5,C,2026-01-17,80.0,60,LogicalRDD_68719476736
6,F,2026-01-16,50.0,70,LogicalRDD_77309411328


In [8]:
with data_provenance_enabled(spark):
    df4 = df2.select("letter")
df4.toPandas()

,letter,_provenance_tag
0,A,LogicalRDD_8589934592
1,B,LogicalRDD_25769803776
2,C,LogicalRDD_42949672960
3,D,LogicalRDD_60129542144
4,E,LogicalRDD_77309411328


In [9]:
df.createOrReplaceTempView("sales")
with data_provenance_enabled(spark):
    res = spark.sql("select product from sales")
res.toPandas()

,product
0,A
1,A
2,A
3,B
4,B
5,C
6,F


In [10]:
df2.createOrReplaceTempView("category")
with data_provenance_enabled(spark):
    res2 = spark.sql("select letter from category")
res2.toPandas()

,letter
0,A
1,B
2,C
3,D
4,E


### Join

In [11]:
df.createOrReplaceTempView("sales")
df2.createOrReplaceTempView("category")
with data_provenance_enabled(spark):
    res = spark.sql("select * from sales s join category c on s.product=c.letter").explain(True)

res

== Parsed Logical Plan ==
'Project [*]
+- 'Join Inner, ('s.product = 'c.letter)
   :- 'SubqueryAlias s
   :  +- 'UnresolvedRelation [sales], [], false
   +- 'SubqueryAlias c
      +- 'UnresolvedRelation [category], [], false

== Analyzed Logical Plan ==
product: string, date: date, price: double, quantity: bigint, letter: string, labell: string
Project [product#0, date#1, price#2, quantity#3L, letter#4, labell#5]
+- Join Inner, (product#0 = letter#4)
   :- SubqueryAlias s
   :  +- SubqueryAlias sales
   :     +- View (`sales`, [product#0, date#1, price#2, quantity#3L])
   :        +- LogicalRDD [product#0, date#1, price#2, quantity#3L], false
   +- SubqueryAlias c
      +- SubqueryAlias category
         +- View (`category`, [letter#4, labell#5])
            +- LogicalRDD [letter#4, labell#5], false

== Optimized Logical Plan ==
Join Inner, (product#0 = letter#4)
:- Filter isnotnull(product#0)
:  +- LogicalRDD [product#0, date#1, price#2, quantity#3L], false
+- Filter isnotnull(letter#

In [12]:

with data_provenance_enabled(spark):
    df5 = df.select("*").join(df2, df.product == df2.letter)

df5.toPandas()

,product,date,price,quantity,letter,labell
0,A,2026-01-15,10.0,90,A,Bike
1,A,2026-01-16,10.0,120,A,Bike
2,A,2026-01-17,5.0,300,A,Bike
3,B,2026-01-15,100.0,20,B,Handball
4,B,2026-01-16,100.0,30,B,Handball
5,C,2026-01-17,80.0,60,C,Bike


In [13]:
with data_provenance_enabled(spark):
    df6 = df.join(df2, df.product == df2.letter).select("product")

df6.toPandas()

,product
0,A
1,A
2,A
3,B
4,B
5,C


In [14]:
with data_provenance_enabled(spark):
    df7 = df.select("*").join(df2, df.product==df2.letter, "outer")
df7.toPandas()

,product,date,price,quantity,letter,labell
0,A,2026-01-15,10.0,90.0,A,Bike
1,A,2026-01-16,10.0,120.0,A,Bike
2,A,2026-01-17,5.0,300.0,A,Bike
3,B,2026-01-15,100.0,20.0,B,Handball
4,B,2026-01-16,100.0,30.0,B,Handball
5,C,2026-01-17,80.0,60.0,C,Bike
6,NaN,None,NaN,NaN,D,Handball
7,NaN,None,NaN,NaN,E,Running
8,F,2026-01-16,50.0,70.0,NaN,NaN


### Where

In [15]:
with data_provenance_enabled(spark):
    df8 = df.select("*").filter("price > 10")
df8.toPandas()
#df3.explain("extended")

,product,date,price,quantity
0,B,2026-01-15,100.0,20
1,B,2026-01-16,100.0,30
2,C,2026-01-17,80.0,60
3,F,2026-01-16,50.0,70


In [16]:
df.createOrReplaceTempView("sales")

with data_provenance_enabled(spark):
    res = spark.sql("select product, price from sales where price>10").toPandas()

res

,product,price
0,B,100.0
1,B,100.0
2,C,80.0
3,F,50.0


### Aggregation

In [17]:
df.createOrReplaceTempView("sales")

with data_provenance_enabled(spark):
    res = spark.sql("select sum(quantity) from sales group by product").toPandas()
res

,sum(quantity)
0,510
1,50
2,60
3,70


In [18]:
# Works only with the select before
# Tag not added without select
with data_provenance_enabled(spark):
    df10 = df.select("*").groupBy("product").agg({"quantity": "sum"})
df10.toPandas()

,product,sum(quantity)
0,A,510
1,B,50
2,C,60
3,F,70


In [19]:
with data_provenance_enabled(spark) :
    df11 = df.withColumn("revenue", df["quantity"] * df["price"]) \
            .groupBy("product") \
            .agg({"revenue": "sum"})
df11.toPandas()

,product,sum(revenue)
0,A,3600.0
1,B,5000.0
2,C,4800.0
3,F,3500.0


In [20]:
# Ne fonctionne qu'avec le select bien présent devant  
# Sinon le tag n'est jamais ajouté
with data_provenance_enabled(spark):
    df10 = df.select("*").groupBy("product").agg({"quantity": "sum"})
df10.toPandas()

,product,sum(quantity)
0,A,510
1,B,50
2,C,60
3,F,70


### Filter / Sort

In [21]:
with data_provenance_enabled(spark):
    df12 = df.select("*").filter("price > 10").sort("price").join(df2, df.product == df2.letter)
df12.toPandas()
#df12.explain(True)

,product,date,price,quantity,letter,labell
0,B,2026-01-15,100.0,20,B,Handball
1,B,2026-01-16,100.0,30,B,Handball
2,C,2026-01-17,80.0,60,C,Bike


In [22]:
df.createOrReplaceTempView("sales")

with data_provenance_enabled(spark):
    res = spark.sql("select * from sales , price from sales where price>10").toPandas()

res

ParseException: 
[PARSE_SYNTAX_ERROR] Syntax error at or near 'sales'. SQLSTATE: 42601 (line 1, pos 33)

== SQL ==
select * from sales , price from sales where price>10
---------------------------------^^^
